pip install vllm pandas matplotlib

In [ ]:
import os

# ---- vLLM notebook / multiprocess safety ----
# 官方支持：关闭 V1，回到 V0 engine
os.environ["VLLM_USE_V1"] = "0"

# 减少 notebook 中 CUDA + multiprocessing 的问题
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# 日志安静一点
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"

import gc
import json
from typing import Dict, List, Tuple

import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

In [ ]:
# ====== paths ======
EVAL_MANIFEST = "/root/data/eval_data_instruct/manifest.json"

MODEL_SFT  = "/root/out/llama-31-8b-instruct-pruned-sft-merged-bf16"
MODEL_KD   = "/root/out/llama-31-8b-instruct-pruned-kd-merged-bf16"
MODEL_BASE = "meta-llama/Llama-3.1-8B-Instruct"

# vLLM runtime knobs
DTYPE = "bfloat16"          # 不支持就改成 "float16"
GPU_MEMORY_UTIL = 0.90
MAX_MODEL_LEN = 2048

# 跑通优先；显存紧张就改小
BATCH_SIZE = 64

# None = 每个数据集全量
N_MAX_PER_DATASET = None

# 候选字母 logprobs top-k
TOP_LOGPROBS = 20

In [ ]:
def load_manifest(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_jsonl(path: str, n_max=None) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
            if n_max is not None and len(rows) >= n_max:
                break
    return rows

def letters(k: int) -> List[str]:
    return [chr(ord("A") + i) for i in range(k)]

def get_letter_token_ids(tok: AutoTokenizer, k: int) -> Dict[str, List[int]]:
    """
    同时考虑 "A" 和 " A"。
    只使用单 token 候选。
    """
    mapping = {}
    for L in letters(k):
        ids = set()
        for s in [L, " " + L]:
            enc = tok.encode(s, add_special_tokens=False)
            if len(enc) == 1:
                ids.add(enc[0])
        mapping[L] = sorted(list(ids))
    return mapping

def chunked(lst, bs):
    for i in range(0, len(lst), bs):
        yield lst[i:i+bs]

def release_vllm(llm_obj):
    del llm_obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
def eval_mc_next_token_logprob(
    model_path: str,
    manifest_path: str,
    dtype: str = "bfloat16",
    gpu_memory_utilization: float = 0.90,
    max_model_len: int = 2048,
    batch_size: int = 64,
    top_logprobs: int = 20,
    n_max_per_dataset=None,
) -> Tuple[pd.DataFrame, LLM]:
    """
    对 manifest 里的每个 jsonl 评测 accuracy：
    - 读取 messages
    - 用 instruct tokenizer 的 chat template 构造 prompt
    - 生成 1 token，取该 token 的 logprobs（top-k）
    - 在候选字母 token (A/B/C/...) 中选 logprob 最大的字母
    """

    manifest = load_manifest(manifest_path)

    print(f"Loading tokenizer: {MODEL_BASE}")
    tok = AutoTokenizer.from_pretrained(
        MODEL_BASE,
        use_fast=True,
        trust_remote_code=True,
    )

    print(f"Loading vLLM model: {model_path}")
    llm = LLM(
        model=model_path,
        dtype=dtype,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        trust_remote_code=True,
    )

    sp = SamplingParams(
        temperature=0.0,
        max_tokens=1,
        logprobs=top_logprobs,
    )

    all_rows = []

    for item in manifest:
        name = item["name"]
        path = item["path"]
        rows = read_jsonl(path, n_max=n_max_per_dataset)
        if len(rows) == 0:
            continue

        prompts = []
        answers = []
        ks = []

        for ex in rows:
            prompt_text = tok.apply_chat_template(
                ex["messages"],
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append(prompt_text)
            answers.append(int(ex["answer_idx"]))
            ks.append(int(ex["num_choices"]))

        correct = 0
        total = 0
        unknown = 0
        known_correct = 0
        known_total = 0

        for batch_rows, batch_prompts, batch_answers, batch_ks in zip(
            chunked(rows, batch_size),
            chunked(prompts, batch_size),
            chunked(answers, batch_size),
            chunked(ks, batch_size),
        ):
            outs = llm.generate(batch_prompts, sp)

            for ex, out, gt, k in zip(batch_rows, outs, batch_answers, batch_ks):
                total += 1

                cand_letters = letters(k)
                letter_ids = get_letter_token_ids(tok, k)

                if not out.outputs:
                    unknown += 1
                    continue

                lp_dict = out.outputs[0].logprobs[0]

                best_letter = None
                best_lp = -1e30

                for L in cand_letters:
                    for tid in letter_ids.get(L, []):
                        if tid in lp_dict:
                            lp = lp_dict[tid].logprob
                            if lp > best_lp:
                                best_lp = lp
                                best_letter = L

                # fallback：直接 parse 生成文本
                if best_letter is None:
                    gen = (out.outputs[0].text or "").strip()
                    if gen:
                        c = gen[0].upper()
                        if c in cand_letters:
                            best_letter = c
                        else:
                            unknown += 1
                            continue
                    else:
                        unknown += 1
                        continue

                pred = ord(best_letter) - ord("A")
                known_total += 1
                if pred == gt:
                    correct += 1
                    known_correct += 1

        acc_strict = correct / max(total, 1)
        acc_known_only = known_correct / max(known_total, 1)

        all_rows.append({
            "model": os.path.basename(model_path.rstrip("/")),
            "dataset": name,
            "n": total,
            "acc": acc_strict,
            "acc_known_only": acc_known_only,
            "unknown": unknown,
            "unknown_rate": unknown / max(total, 1),
        })

        print(
            f"{name:14s}  "
            f"acc={acc_strict:.4f}  "
            f"acc_known={acc_known_only:.4f}  "
            f"n={total}  "
            f"unknown={unknown} ({unknown/max(total,1):.2%})"
        )

    df = pd.DataFrame(all_rows)

    if len(df) > 0:
        total_n = df["n"].sum()
        overall = (df["acc"] * df["n"]).sum() / max(total_n, 1)

        known_n = (df["n"] - df["unknown"]).sum()
        overall_known = (
            (df["acc_known_only"] * (df["n"] - df["unknown"])).sum() / max(known_n, 1)
        )

        df_overall = pd.DataFrame([{
            "model": os.path.basename(model_path.rstrip("/")),
            "dataset": "overall_micro",
            "n": int(total_n),
            "acc": float(overall),
            "acc_known_only": float(overall_known),
            "unknown": int(df["unknown"].sum()),
            "unknown_rate": float(df["unknown"].sum() / max(total_n, 1)),
        }])

        df = pd.concat([df, df_overall], ignore_index=True)

    return df, llm

In [ ]:
df_sft, llm_sft = eval_mc_next_token_logprob(
    model_path=MODEL_SFT,
    manifest_path=EVAL_MANIFEST,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
    batch_size=BATCH_SIZE,
    top_logprobs=TOP_LOGPROBS,
    n_max_per_dataset=N_MAX_PER_DATASET,
)

df_sft

In [ ]:
release_vllm(llm_sft)
print("Released SFT model.")

In [ ]:
df_kd, llm_kd = eval_mc_next_token_logprob(
    model_path=MODEL_KD,
    manifest_path=EVAL_MANIFEST,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
    batch_size=BATCH_SIZE,
    top_logprobs=TOP_LOGPROBS,
    n_max_per_dataset=N_MAX_PER_DATASET,
)

df_kd

In [ ]:
release_vllm(llm_kd)
print("Released KD model.")

In [ ]:
df_base, llm_base = eval_mc_next_token_logprob(
    model_path=MODEL_BASE,
    manifest_path=EVAL_MANIFEST,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    max_model_len=MAX_MODEL_LEN,
    batch_size=BATCH_SIZE,
    top_logprobs=TOP_LOGPROBS,
    n_max_per_dataset=N_MAX_PER_DATASET,
)

df_base

In [ ]:
release_vllm(llm_base)
print("Released BASE model.")

In [ ]:
df_all = pd.concat([df_base, df_sft, df_kd], ignore_index=True)

order = ["hellaswag", "arc_challenge", "winogrande", "truthfulqa_mc", "overall_micro"]

df_plot = df_all.copy()
df_plot["dataset"] = pd.Categorical(df_plot["dataset"], categories=order, ordered=True)
df_plot = df_plot.sort_values(["dataset", "model"])

datasets = [d for d in order if d in df_plot["dataset"].unique().tolist()]
pv = df_plot.pivot(index="dataset", columns="model", values="acc").loc[datasets]

ax = pv.plot(kind="bar", figsize=(10, 5))
ax.set_ylabel("Accuracy (strict)")
ax.set_xlabel("Dataset")
ax.set_title("Base vs SFT vs KD (pre-quant, next-token logprob MC accuracy)")
ax.legend(title="Model", loc="best")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()